# 08. Saving & Loading Models — Keras

A trained model is worthless if you can't reuse it. We cover the three Keras persistence options: the modern **`.keras`** file, **weights-only**, and the **SavedModel** export for serving.

**Dataset:** `loan_prediction_data.csv` — 614 loan applications, 11 pre-normalised features, binary target `Loan_Status` (1 = approved).

---

## 1. Setup & imports

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)          # reproducible weights + shuffling
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

TensorFlow 2.21.0 | Keras 3.14.0


## 2. Load the data

In [2]:
import os, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

# Robust load: works whether the notebook runs from its own folder or the parent
CSV = "loan_prediction_data.csv"
if not os.path.exists(CSV):
    CSV = os.path.join("..", "loan_prediction_data.csv")

df = pd.read_csv(CSV)
print("Shape:", df.shape)
df.head()

Shape: (614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,0.0,0.0,0.000000,1.0,0.0,0.070489,0.000000,0.198860,0.74359,1.0,1.0,1.0
1,LP001003,0.0,1.0,0.333333,1.0,0.0,0.054830,0.036192,0.172214,0.74359,1.0,0.0,0.0
2,LP001005,0.0,1.0,0.000000,1.0,1.0,0.035250,0.000000,0.082489,0.74359,1.0,1.0,1.0
3,LP001006,0.0,1.0,0.000000,0.0,0.0,0.030093,0.056592,0.160637,0.74359,1.0,1.0,1.0
4,LP001008,0.0,0.0,0.000000,1.0,0.0,0.072356,0.000000,0.191027,0.74359,1.0,1.0,1.0


## 3. Prepare features & train/test split

The features are already scaled to `[0,1]`, so we only drop the ID and split. We use a **stratified** split so the approval rate matches in train and test.

In [3]:
# Features are ALREADY normalised to [0,1] in this dataset, so no scaling needed.
# Drop the ID column and separate the target.
X = df.drop(columns=["Loan_ID", "Loan_Status"]).values.astype("float32")
y = df["Loan_Status"].values.astype("float32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
N_FEATURES = X_train.shape[1]
print("Train:", X_train.shape, " Test:", X_test.shape, " Features:", N_FEATURES)
print("Train positive rate: %.3f  Test positive rate: %.3f" % (y_train.mean(), y_test.mean()))

Train: (491, 11)  Test: (123, 11)  Features: 11
Train positive rate: 0.686  Test positive rate: 0.691


In [4]:
def plot_history(hist, title=""):
    """Plot train (solid) vs test/val (dashed) loss and accuracy over epochs."""
    h = hist.history
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h["loss"], label="train")
    ax[0].plot(h["val_loss"], "--", label="test")
    ax[0].set_title(title + " — Loss"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss")
    ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(h["accuracy"], label="train")
    ax[1].plot(h["val_accuracy"], "--", label="test")
    ax[1].set_title(title + " — Accuracy"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy")
    ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 4. Train a model to save

In [5]:
keras.utils.set_random_seed(42)
model = keras.Sequential([
    keras.Input(shape=(N_FEATURES,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1,  activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, epochs=60, batch_size=32, verbose=0)
ref_pred = model.predict(X_test[:5], verbose=0).ravel()
print("Reference predictions:", np.round(ref_pred, 5))

Reference predictions: [0.16155 0.81844 0.79023 0.82677 0.8069 ]


## 5. Option A — full model as `.keras` (recommended)

One file holds architecture + weights + optimizer state. Reload with `keras.models.load_model` and you can predict *or* keep training.

In [6]:
model.save("loan_model.keras")
reloaded = keras.models.load_model("loan_model.keras")
new_pred = reloaded.predict(X_test[:5], verbose=0).ravel()
print("Reloaded predictions :", np.round(new_pred, 5))
print("Identical to original?", np.allclose(ref_pred, new_pred))

Reloaded predictions : [0.16155 0.81844 0.79023 0.82677 0.8069 ]
Identical to original? True


## 6. Option B — weights only

Saves just the numbers. You must rebuild the *same architecture* first, then `load_weights`. Useful for checkpoints and sharing weights.

In [7]:
model.save_weights("loan_weights.weights.h5")

fresh = keras.Sequential([
    keras.Input(shape=(N_FEATURES,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1,  activation="sigmoid"),
])
fresh.load_weights("loan_weights.weights.h5")
w_pred = fresh.predict(X_test[:5], verbose=0).ravel()
print("Weights-loaded preds :", np.round(w_pred, 5))
print("Identical to original?", np.allclose(ref_pred, w_pred))

Weights-loaded preds : [0.16155 0.81844 0.79023 0.82677 0.8069 ]
Identical to original? True


## 7. Option C — SavedModel export (for serving)

`model.export` writes a TensorFlow SavedModel directory — the format used by TF Serving / TFLite. It is inference-only (no further training).

In [8]:
model.export("loan_savedmodel")     # creates a directory
import os
print("SavedModel files:", os.listdir("loan_savedmodel"))

infer = keras.layers.TFSMLayer("loan_savedmodel", call_endpoint="serve")
sm_out = infer(X_test[:5])
sm_pred = np.array(sm_out).ravel() if not isinstance(sm_out, dict) else np.array(list(sm_out.values())[0]).ravel()
print("SavedModel preds     :", np.round(sm_pred, 5))

INFO:tensorflow:Assets written to: loan_savedmodel\assets


INFO:tensorflow:Assets written to: loan_savedmodel\assets


Saved artifact at 'loan_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 11), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1237614842640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1237614843792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1237614843408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1237614844176: TensorSpec(shape=(), dtype=tf.resource, name=None)


SavedModel files: ['assets', 'fingerprint.pb', 'saved_model.pb', 'variables']
SavedModel preds     : [0.16155 0.81844 0.79023 0.82677 0.8069 ]


## Takeaways
| Method | File(s) | Contains | Reuse |
|---|---|---|---|
| `.keras` | one file | arch + weights + optimizer | predict **and** resume training |
| weights-only | `.weights.h5` | weights only | must rebuild arch first |
| SavedModel (`export`) | directory | inference graph | serving / TFLite, no training |

- Prefer **`.keras`** for everyday checkpoints — it round-trips exactly (we verified `np.allclose`).
- Use **weights-only** for lightweight checkpoints during long training.
- Use **SavedModel** when deploying to TF Serving or mobile.
- **Next:** `09_comparison_keras` — put every technique together.